In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 49.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=b54709f23f3ee9fa74248c9d1c39fab600db306f842a948dc1b35ad7a87b4060
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# Quantum random number generation and qubit encode/measure.
# These helpers are used by all agents in the protocol.

backend = BasicSimulator()

def quantum_random_bit():
    # Generate a truly random bit by preparing the superposition state |+> = 1/sqrt(2) * (|0> + |1>) via a Hadamard gate on |0>, then measuring.
    # The outcome is 0 or 1 with equal probability 1/2.

    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    # Generate a list of n quantum random bits.
    return [quantum_random_bit() for _ in range(n)]


def encode_qubit(bit, basis):
    # Encode a classical bit into a qubit using the given basis.
    # Returns a QuantumCircuit representing the prepared qubit.
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc


def measure_qubit(qc, basis):
    # Measure a qubit circuit in the given basis.
    # Returns the measured bit (0 or 1).
    qc_m = qc.copy()
    if basis == 1:
        qc_m.h(0)
    qc_m.measure_all()
    compiled = transpile(qc_m, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [4]:
# Total number of qubits to send
N = 100

# Alice generates her secret random bit values
alice_bits  = quantum_random_bits(N)

# Alice generates her random basis choices
alice_bases = quantum_random_bits(N)

# Alice encodes each bit into a qubit and sends it
qubits_from_alice = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

print("ALICE")
print(f"Bits sent (first 20): {alice_bits[:20]}")
print(f"Bases chosen (first 20): {alice_bases[:20]}")
print(f"Alice has encoded and sent {N} qubits.")

ALICE
Bits sent (first 20): [1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0]
Bases chosen (first 20): [1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0]
Alice has encoded and sent 100 qubits.


In [5]:
# Eve picks a random basis for each qubit she intercepts
eve_bases = quantum_random_bits(N)
eve_results = []

# Eve intercepts each qubit, measures it, and resends a fresh qubit
qubits_from_eve = []

for i in range(N):
    # Eve measures Alice's qubit in her chosen basis
    eve_bit = measure_qubit(qubits_from_alice[i], eve_bases[i])
    eve_results.append(eve_bit)

    # Eve re-encodes a fresh qubit based on what she measured
    # and forwards it to Bob (in her own basis, not Alice's)
    qubits_from_eve.append(encode_qubit(eve_bit, eve_bases[i]))

print("EVE (Attacker)")
print(f"Eve's bases (first 20): {eve_bases[:20]}")
print(f"Eve's results (first 20): {eve_results[:20]}")
print(f"Eve has intercepted and resent all {N} qubits.")

# Count how often Eve guessed Alice's basis correctly
correct_guesses = sum(eve_bases[i] == alice_bases[i] for i in range(N))
print(f"Eve guessed the correct basis {correct_guesses}/{N} times ({100*correct_guesses/N:.1f}%)")

EVE (Attacker)
Eve's bases (first 20): [0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0]
Eve's results (first 20): [1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0]
Eve has intercepted and resent all 100 qubits.
Eve guessed the correct basis 51/100 times (51.0%)


In [6]:
# Bob generates his random basis choices, independent of Alice and Eve
bob_bases = quantum_random_bits(N)

# Bob measures the qubits he received (which came from Eve, not Alice)
bob_results = [measure_qubit(qubits_from_eve[i], bob_bases[i]) for i in range(N)]

print("BOB")
print(f"Bases chosen (first 20): {bob_bases[:20]}")
print(f"Results (first 20): {bob_results[:20]}")
print(f"Bob has measured all {N} qubits (received via Eve).")

BOB
Bases chosen (first 20): [1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1]
Results (first 20): [0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0]
Bob has measured all 100 qubits (received via Eve).


In [7]:
# Sifting
sifted_alice = []
sifted_bob = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:   # Bases match: keep this bit
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print("SIFTING")
print(f"Total qubits sent: {N}")
print(f"Bits retained after sift: {len(sifted_alice)} ({100*len(sifted_alice)/N:.1f}%)")
print(f"Sifted Alice (first 20): {sifted_alice[:20]}")
print(f"Sifted Bob (first 20): {sifted_bob[:20]}")

SIFTING
Total qubits sent: 100
Bits retained after sift: 54 (54.0%)
Sifted Alice (first 20): [1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
Sifted Bob (first 20): [0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0]


In [8]:
# Error Checking
SAMPLE_SIZE = max(1, len(sifted_alice) // 5)  # Sacrifice ~20% for error checking
THRESHOLD = 0.10                             # Flag if error rate exceeds 10%

sample_alice = sifted_alice[:SAMPLE_SIZE]
sample_bob = sifted_bob[:SAMPLE_SIZE]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / SAMPLE_SIZE

print("ERROR CHECKING")
print(f"Check sample size: {SAMPLE_SIZE} bits")
print(f"Errors found: {errors}")
print(f"Error rate: {error_rate:.1%} (expected ~25% with Eve intercepting all qubits)")
print()

if error_rate > THRESHOLD:
    print(f"Error rate {error_rate:.1%} exceeds threshold {THRESHOLD:.0%}.")
    print("High error rate. Possible attacker.")
    final_alice_key = []
    final_bob_key   = []
else:
    print("Error rate within bounds.")
    final_alice_key = sifted_alice[SAMPLE_SIZE:]
    final_bob_key   = sifted_bob[SAMPLE_SIZE:]

ERROR CHECKING
Check sample size: 10 bits
Errors found: 5
Error rate: 50.0% (expected ~25% with Eve intercepting all qubits)

Error rate 50.0% exceeds threshold 10%.
High error rate. Possible attacker.


In [9]:
# Summary and Checks
print("PROTOCOL SUMMARY")
print(f"Qubits sent by Alice: {N}")
print(f"Qubits intercepted by Eve: {N} (100% intercept rate)")
print(f"Eve's basis correct: {correct_guesses}/{N} ({100*correct_guesses/N:.1f}%)")
print(f"Bits after sifting: {len(sifted_alice)}")
print(f"Errors in sifted key: {sum(a != b for a,b in zip(sifted_alice, sifted_bob))}")
print(f"Overall sifted error rate: {sum(a != b for a,b in zip(sifted_alice, sifted_bob))/len(sifted_alice):.1%}")
print(f"Check sample error rate: {error_rate:.1%}")
print()
print("RESULT: Eve was" + (" DETECTED." if error_rate > THRESHOLD else " NOT detected (statistically unlikely but possible with small N)."))
print()
print("Why ~25% errors?")
print("Eve guesses wrong basis:   ~50% of qubits")
print("Wrong-basis qubit gives Bob wrong bit: ~50% of those")
print("=> 0.50 * 0.50 = 0.25 = 25% error rate in the sifted key")

PROTOCOL SUMMARY
Qubits sent by Alice: 100
Qubits intercepted by Eve: 100 (100% intercept rate)
Eve's basis correct: 51/100 (51.0%)
Bits after sifting: 54
Errors in sifted key: 15
Overall sifted error rate: 27.8%
Check sample error rate: 50.0%

RESULT: Eve was DETECTED.

Why ~25% errors?
Eve guesses wrong basis:   ~50% of qubits
Wrong-basis qubit gives Bob wrong bit: ~50% of those
=> 0.50 * 0.50 = 0.25 = 25% error rate in the sifted key
